In [ ]:
# ============================================================
# MODULE 4: DATA CLEANING AND REPORTING AUTOMATION
# THIRANEX DATA SCIENCE INTERNSHIP
# ============================================================

import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------
# 1. CREATE SAMPLE RAW SALES DATASET
# ------------------------------------------------------------

np.random.seed(42)

n = 120

categories = ["Technology", "Furniture", "Office Supplies"]
regions = ["North", "South", "East", "West"]
segments = ["Consumer", "Corporate", "Home Office"]
products = ["Laptop", "Chair", "Printer", "Table", "Phone", "Notebook"]
ship_modes = ["Standard", "Second Class", "First Class"]

sales_df = pd.DataFrame({
    "Order_ID": range(1001, 1001 + n),
    "Order_Date": pd.date_range("2025-01-01", periods=n, freq="D"),
    "Category": np.random.choice(categories, n),
    "Product": np.random.choice(products, n),
    "Region": np.random.choice(regions, n),
    "Segment": np.random.choice(segments, n),
    "Ship_Mode": np.random.choice(ship_modes, n),
    "Quantity": np.random.randint(1, 10, n),
    "Sales": np.random.uniform(100, 5000, n).round(2)
})

sales_df["Profit"] = (
    sales_df["Sales"] * np.random.uniform(0.05, 0.30, n)
).round(2)

# ------------------------------------------------------------
# 2. INTRODUCE SOME DATA QUALITY ISSUES
# ------------------------------------------------------------

# Add duplicate records
duplicate_rows = sales_df.iloc[:3].copy()
sales_df = pd.concat(
    [sales_df, duplicate_rows],
    ignore_index=True
)

# Add missing values
sales_df.loc[5, "Product"] = np.nan
sales_df.loc[10, "Region"] = np.nan
sales_df.loc[15, "Sales"] = np.nan
sales_df.loc[20, "Profit"] = np.nan

# Add invalid values
sales_df.loc[25, "Quantity"] = 0
sales_df.loc[30, "Sales"] = -500
sales_df.loc[35, "Profit"] = -100

# ------------------------------------------------------------
# 3. INITIAL DATA INSPECTION
# ------------------------------------------------------------

print("=" * 60)
print("DATA CLEANING AND REPORTING AUTOMATION")
print("=" * 60)

print("\nInitial Dataset Shape:", sales_df.shape)

print("\nFirst 5 Records:")
display(sales_df.head())

print("\nMissing Values Before Cleaning:")
print(sales_df.isnull().sum())

duplicates_before = sales_df.duplicated().sum()

print("\nDuplicate Records Before Cleaning:", duplicates_before)

# ------------------------------------------------------------
# 4. DATA CLEANING
# ------------------------------------------------------------

# Make a copy for cleaning
cleaned_df = sales_df.copy()

# Convert date column
cleaned_df["Order_Date"] = pd.to_datetime(
    cleaned_df["Order_Date"],
    errors="coerce"
)

# Remove duplicate records
cleaned_df = cleaned_df.drop_duplicates()

# Fill missing numerical values with median
numerical_columns = ["Quantity", "Sales", "Profit"]

for column in numerical_columns:
    cleaned_df[column] = cleaned_df[column].fillna(
        cleaned_df[column].median()
    )

# Fill missing categorical values with Unknown
categorical_columns = [
    "Category",
    "Product",
    "Region",
    "Segment",
    "Ship_Mode"
]

for column in categorical_columns:
    cleaned_df[column] = cleaned_df[column].fillna("Unknown")

# ------------------------------------------------------------
# 5. REMOVE INVALID RECORDS
# ------------------------------------------------------------

# Remove zero or negative quantity
cleaned_df = cleaned_df[
    cleaned_df["Quantity"] > 0
]

# Remove negative sales
cleaned_df = cleaned_df[
    cleaned_df["Sales"] >= 0
]

# Remove negative profit
cleaned_df = cleaned_df[
    cleaned_df["Profit"] >= 0
]

# ------------------------------------------------------------
# 6. CHECK CLEANED DATA
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CLEANED DATASET")
print("=" * 60)

print("\nFinal Dataset Shape:", cleaned_df.shape)

print("\nMissing Values After Cleaning:")
print(cleaned_df.isnull().sum())

print(
    "\nDuplicate Records After Cleaning:",
    cleaned_df.duplicated().sum()
)

print("\nCleaned Dataset Preview:")
display(cleaned_df.head())

# ------------------------------------------------------------
# 7. GENERATE AUTOMATED SUMMARY REPORT
# ------------------------------------------------------------

total_records = len(cleaned_df)
total_sales = cleaned_df["Sales"].sum()
total_profit = cleaned_df["Profit"].sum()
total_quantity = cleaned_df["Quantity"].sum()

average_sales = cleaned_df["Sales"].mean()
average_profit = cleaned_df["Profit"].mean()

duplicates_removed = duplicates_before - cleaned_df.duplicated().sum()

summary_report = pd.DataFrame({
    "Metric": [
        "Total Records",
        "Total Sales",
        "Total Profit",
        "Total Quantity",
        "Average Sales",
        "Average Profit",
        "Duplicate Records Removed",
        "Missing Values Remaining"
    ],
    "Value": [
        total_records,
        round(total_sales, 2),
        round(total_profit, 2),
        total_quantity,
        round(average_sales, 2),
        round(average_profit, 2),
        duplicates_removed,
        int(cleaned_df.isnull().sum().sum())
    ]
})

print("\n" + "=" * 60)
print("AUTOMATED SUMMARY REPORT")
print("=" * 60)

display(summary_report)

# ------------------------------------------------------------
# 8. SAVE CLEANED DATASET
# ------------------------------------------------------------

cleaned_df.to_csv(
    "cleaned_sales_dataset.csv",
    index=False
)

# ------------------------------------------------------------
# 9. SAVE AUTOMATED SUMMARY REPORT
# ------------------------------------------------------------

summary_report.to_csv(
    "automated_summary_report.csv",
    index=False
)

# ------------------------------------------------------------
# 10. GENERATE ADDITIONAL REPORT
# ------------------------------------------------------------

category_report = cleaned_df.groupby(
    "Category"
).agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Total_Quantity=("Quantity", "sum")
).reset_index()

category_report.to_csv(
    "cleaned_category_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 11. FINAL VALIDATION
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL DATA VALIDATION")
print("=" * 60)

print("Missing Values:", cleaned_df.isnull().sum().sum())
print("Duplicate Records:", cleaned_df.duplicated().sum())
print("Negative Sales:", (cleaned_df["Sales"] < 0).sum())
print("Negative Profit:", (cleaned_df["Profit"] < 0).sum())
print("Invalid Quantity:", (cleaned_df["Quantity"] <= 0).sum())

# ------------------------------------------------------------
# 12. DISPLAY GENERATED FILES
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("GENERATED OUTPUT FILES")
print("=" * 60)

output_files = [
    "cleaned_sales_dataset.csv",
    "automated_summary_report.csv",
    "cleaned_category_summary.csv"
]

for file in output_files:
    print("✓", file)

# ------------------------------------------------------------
# 13. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("MODULE 4 COMPLETED SUCCESSFULLY")
print("=" * 60)

print(f"\nTotal Records After Cleaning : {total_records}")
print(f"Total Sales                 : {total_sales:.2f}")
print(f"Total Profit                : {total_profit:.2f}")
print(f"Average Sales               : {average_sales:.2f}")
print(f"Average Profit              : {average_profit:.2f}")
print(f"Duplicate Records Removed   : {duplicates_removed}")

print("\nData cleaning and reporting automation completed successfully!")